# Pro-Cure · Notebook 01 · Data Loading & Schema

**Day 2 — Updated after full dataset confirmed**


The full OCDS download is a **relational star schema** — 8 separate CSV files
joined on `ocid`. This notebook loads, joins, and prepares the analytical table.

### File structure
```
data/raw/full/
    main.csv               ← 33,689 tenders (core table)
    awards.csv             ← 9,090 awards + contract values
    awards_suppliers.csv   ← 9,090 supplier names
    contracts.csv          ← 1,223 signed contracts
    parties.csv            ← 9,090 company details
    tender_tenderers.csv   ← 3,388 bidder records
    tender_documents.csv   ← 37,073 document links
    contracts_documents.csv← 1,405 contract documents
```

### Join key
`main.ocid` ↔ `awards.main_ocid` ↔ `awards_suppliers.main_ocid`

## 0. Imports & paths

In [ ]:
import os
import glob
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

BASE_DIR      = os.path.abspath(os.path.join(os.getcwd(), ".."))
RAW_DIR       = os.path.join(BASE_DIR, "data", "raw")
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
HEALTH_DIR    = os.path.join(BASE_DIR, "data", "health_sector")

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(HEALTH_DIR,    exist_ok=True)

# Expected location of the full CSV download
FULL_DIR = os.path.join(RAW_DIR, "full")

print("✅ Imports OK")
print(f"   Raw dir:       {RAW_DIR}")
print(f"   Processed dir: {PROCESSED_DIR}")
print(f"   Health dir:    {HEALTH_DIR}")
print(f"   Full CSV dir:  {FULL_DIR}")


## 1. Confirmed schema — relational star model

> ⭐ **Key insight:** `tender_value_amount` in `main.csv` is **all zeros**.
> Real contract values are in `awards.csv → value_amount`.
> Join `main.ocid → awards.main_ocid` to access them.


In [ ]:
# Confirmed column schema from live data — May 2026
SCHEMA = {
    "main.csv": {
        "join_key":    "ocid",
        "rows":        "33,689",
        "key_fields": {
            "ocid":                          "⭐ Join key to all other tables",
            "buyer_name":                    "⭐ Government entity (100% populated)",
            "date":                          "⭐ Tender date — ISO format (100% populated)",
            "tender_id":                     "⭐ Tender number (100% populated)",
            "tender_description":            "⭐ Full description for NLP (100% populated)",
            "tender_procurementMethod":      "⭐ open/direct/limited/selective",
            "tender_mainProcurementCategory":"Commodity category (79% populated)",
            "tender_province":               "Province (100% populated)",
            "tender_value_amount":           "⚠️  ALL ZEROS — ignore this column",
        }
    },
    "awards.csv": {
        "join_key":    "main_ocid → main.ocid",
        "rows":        "9,090",
        "key_fields": {
            "main_ocid":   "⭐ Join key back to main",
            "value_amount":"⭐ REAL contract value (use this for all value analysis)",
            "status":      "Award status",
        }
    },
    "awards_suppliers.csv": {
        "join_key":    "main_ocid → main.ocid",
        "rows":        "9,090",
        "key_fields": {
            "main_ocid":   "⭐ Join key back to main",
            "name":        "⭐ Supplier name — use for network analysis",
            "awards_id":   "Links to awards.csv",
        }
    },
    "parties.csv": {
        "join_key":    "main_ocid → main.ocid",
        "rows":        "9,090",
        "key_fields": {
            "identifier_legalName": "⭐ Legal entity name — dedup supplier names",
            "roles":                "buyer / supplier / tenderer",
            "contactPoint_email":   "Contact details",
        }
    },
}

for table, info in SCHEMA.items():
    print(f"\n  📄 {table}  ({info['rows']} rows)")
    print(f"     Join: {info['join_key']}")
    for col, desc in info['key_fields'].items():
        print(f"     {col:<40} {desc}")


## 2. Load all tables



In [ ]:
def load_table(name, directory=FULL_DIR):
    """Load a single OCDS table by filename."""
    path = os.path.join(directory, name)
    if not os.path.exists(path):
        # Also search recursively in RAW_DIR
        matches = glob.glob(os.path.join(RAW_DIR, "**", name), recursive=True)
        if matches:
            path = matches[0]
        else:
            print(f"  ⚠️  {name} not found — searched in {directory}")
            return pd.DataFrame()
    df = pd.read_csv(path, low_memory=False)
    print(f"  ✅ {name:<35} {len(df):>8,} rows × {df.shape[1]} cols")
    return df

print("Loading OCDS tables...\n")
main_df         = load_table("main.csv")
awards_df       = load_table("awards.csv")
suppliers_df    = load_table("awards_suppliers.csv")
contracts_df    = load_table("contracts.csv")
parties_df      = load_table("parties.csv")
tenderers_df    = load_table("tender_tenderers.csv")


## 3. Build the analytical table

Join `main` → `awards` → `awards_suppliers` on `ocid`.
This is the master table used for all anomaly detection.


In [ ]:
if not main_df.empty:

    # Step 1: Join awards (contract values)
    df = main_df.merge(
        awards_df[["main_ocid", "value_amount", "status"]].rename(
            columns={"value_amount": "award_value", "status": "award_status"}),
        left_on="ocid", right_on="main_ocid",
        how="left"
    )

    # Step 2: Join supplier names
    df = df.merge(
        suppliers_df[["main_ocid", "name"]].rename(columns={"name": "supplier_name"}),
        left_on="ocid", right_on="main_ocid",
        how="left"
    )

    # Step 3: Parse dates and extract year
    df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True)
    df["year"] = df["date"].dt.year.astype("Int64")

    # Step 4: Clean award values — flag implausible outliers (> R10bn)
    df["award_value"] = pd.to_numeric(df["award_value"], errors="coerce")
    df["award_value_clean"] = df["award_value"].where(
        (df["award_value"] > 0) & (df["award_value"] < 10_000_000_000), np.nan)

    # Step 5: Normalise names to uppercase for consistent matching
    for col in ["buyer_name", "supplier_name"]:
        if col in df.columns:
            df[col] = df[col].str.strip().str.upper()

    print(f"✅ Analytical table: {len(df):,} rows × {df.shape[1]} cols")
    print(f"   Tenders with award value:    {df['award_value_clean'].notna().sum():,}")
    print(f"   Tenders with supplier name:  {df['supplier_name'].notna().sum():,}")
    print(f"   Date range: {df['year'].min()} – {df['year'].max()}")

    # Summary stats on award values
    av = df["award_value_clean"].dropna()
    print(f"\nAward value stats ({len(av):,} records):")
    print(f"   Min:    R{av.min():>15,.0f}")
    print(f"   Median: R{av.median():>15,.0f}")
    print(f"   Mean:   R{av.mean():>15,.0f}")
    print(f"   Max:    R{av.max():>15,.0f}")
else:
    print("❌ main_df is empty — check file paths above")
    df = pd.DataFrame()


## 4. Year distribution

In [ ]:
if not df.empty:
    print("Tenders by year:")
    for yr, c in df["year"].value_counts().sort_index().items():
        bar = "█" * int(c / 300)
        print(f"  {yr}: {c:>6,}  {bar}")


## 5. Award value distribution — first corruption signal

Contract values clustered just below R500k and R200k thresholds indicate
contract splitting — the core Tembisa pattern.


In [ ]:
if not df.empty:
    av = df["award_value_clean"].dropna()

    THRESHOLDS = [0, 200_000, 500_000, 1_000_000, 10_000_000, 100_000_000, float("inf")]
    LABELS = ["< R200k (RFQ)", "R200k–R500k (HSP)", "R500k–R1m",
              "R1m–R10m", "R10m–R100m", "> R100m"]

    buckets = pd.cut(av, bins=THRESHOLDS, labels=LABELS)
    print(f"Award value distribution ({len(av):,} records):")
    print(f"  {'Bucket':<25} {'Count':>8}  {'%':>7}  Bar")
    print("  " + "-" * 60)
    for label, count in buckets.value_counts().sort_index().items():
        pct = 100 * count / len(av)
        bar = "█" * int(pct / 2)
        print(f"  {label:<25} {count:>8,}  {pct:>6.1f}%  {bar}")

    # Splitting signals
    near_500k = df[(df["award_value_clean"] >= 475_000) & (df["award_value_clean"] < 500_000)]
    near_200k = df[(df["award_value_clean"] >= 190_000) & (df["award_value_clean"] < 200_000)]
    print(f"\n  ⭐ Within 5% below R500k threshold: {len(near_500k):,} contracts — splitting signal")
    print(f"  ⭐ Within 5% below R200k threshold: {len(near_200k):,} contracts — splitting signal")

    if len(near_500k) > 0:
        print(f"\n  Top buyers near R500k threshold:")
        for b, c in near_500k["buyer_name"].value_counts().head(10).items():
            print(f"    {c:>3,}  {b}")


## 6. Procurement method breakdown — second corruption signal

In [ ]:
if not df.empty:
    methods = df["tender_procurementMethod"].value_counts()
    total = methods.sum()
    print(f"Procurement methods ({total:,} records):\n")
    print(f"  {'Method':<45} {'Count':>8}  {'%':>7}")
    print("  " + "-" * 65)
    for method, count in methods.items():
        pct = 100 * count / total
        flag = " ← ⚠️  DEVIATION" if any(
            x in str(method).lower() for x in ["direct","limited","selective"]) else ""
        print(f"  {str(method):<45} {count:>8,}  ({pct:.1f}%){flag}")


## 7. Health sector keywords

> ⚠️  SA naming convention uses `GAUTENG - HEALTH`, `KWAZULU NATAL - HEALTH` etc.
> NOT the full department name format. Keywords are calibrated to match this.


In [ ]:
# Calibrated to actual buyer_name values in this dataset
HEALTH_INCLUDE = [
    "health", "hospital", "tembisa",
    "chris hani", "charlotte maxeke", "helen joseph",
    "groote schuur", "tygerberg", "inkosi albert",
    "pelonomi", "mankweng", "national health laboratory",
]

# False positives to exclude
HEALTH_EXCLUDE = [
    "hospitality",
    "mine health and safety",
    "health and welfare seta",
    "health and welfare sector",
]

print(f"✅ Include keywords: {len(HEALTH_INCLUDE)}")
print(f"✅ Exclude keywords: {len(HEALTH_EXCLUDE)}")


## 8. Health sector filter

In [ ]:
if not df.empty:
    inc = df["buyer_name"].str.lower().str.contains(
        "|".join(HEALTH_INCLUDE), na=False, regex=True)
    exc = df["buyer_name"].str.lower().str.contains(
        "|".join(HEALTH_EXCLUDE), na=False, regex=True)
    health_df = df[inc & ~exc].copy()

    pct = 100 * len(health_df) / len(df)
    print(f"✅ Health sector: {len(health_df):,} records ({pct:.1f}% of total)")
    print(f"   With award value:   {health_df['award_value_clean'].notna().sum():,}")
    print(f"   With supplier name: {health_df['supplier_name'].notna().sum():,}")

    print(f"\nTop health sector buyers:")
    print("-" * 55)
    for buyer, count in health_df["buyer_name"].value_counts().head(15).items():
        marker = " ← 🎯 TEMBISA" if "TEMBISA" in str(buyer) else ""
        print(f"  {count:>5,}  {buyer}{marker}")

    # Health deviations
    deviations = health_df[
        health_df["tender_procurementMethod"].isin(["direct", "limited", "selective"])
    ]
    print(f"\n⚠️  Health deviations (direct/limited/selective): {len(deviations):,}")
    if len(deviations) > 0:
        for b, c in deviations["buyer_name"].value_counts().items():
            print(f"  {c:>3,}  {b}")
else:
    health_df = pd.DataFrame()


## 9. Save outputs